# NeuroX v4 - Fast GPU Training (Google Colab T4)

## Train ALL 17 models in 5-15 minutes instead of 12+ hours!

**How it works:**
- Colab gives you a free NVIDIA T4 GPU (8.1 TFLOPS vs ~0.1 on CPU)
- GPU batch size = 256 (vs 32 on CPU) = 8x fewer gradient steps per epoch
- Total speedup: ~80-100x faster than your Windows trading PC

**Instructions:**
1. Make sure GPU is enabled: Runtime -> Change runtime type -> T4 GPU
2. Click "Runtime -> Run all" (or run cells one by one)
3. Wait ~10 minutes for all 17 models to train
4. Download the checkpoints.zip file at the end
5. Unzip into your trading PC: neurox_v4/checkpoints/

**No CSV upload needed** - data is downloaded from Yahoo Finance automatically!


In [ ]:
#@title Step 0: Verify GPU is available (MUST show T4)
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
    print()
    print("GPU is ready! Training will be FAST.")
else:
    print("ERROR: No GPU detected!")
    print("Go to: Runtime -> Change runtime type -> T4 GPU")
    print("Then restart and re-run this notebook.")
    raise RuntimeError("No GPU - cannot proceed")


In [ ]:
#@title Step 1: Install dependencies and clone repo (~2 min)
import os

# Clone the repo if not already done
if not os.path.exists("/content/Claude"):
    !git clone https://github.com/gagandocx/Claude.git /content/Claude
    print("Repo cloned!")
else:
    # Pull latest changes
    %cd /content/Claude
    !git pull
    print("Repo updated!")

# Install dependencies
!pip install -q torch scikit-learn==1.9.0 lightgbm catboost xgboost     yfinance ta pandas numpy tqdm joblib hmmlearn scipy

print()
print("All dependencies installed!")


In [ ]:
#@title Step 2: Train ALL 17 models on GPU (~5-15 min)
%cd /content/Claude/NeuroX/neurox_v4
!python train_gpu_fast.py


In [ ]:
#@title Step 3: Download trained checkpoints to your PC
import shutil
from google.colab import files

checkpoint_dir = "/content/Claude/NeuroX/neurox_v4/checkpoints"

# List all saved files
print("Saved checkpoints:")
for f in sorted(os.listdir(checkpoint_dir)):
    size = os.path.getsize(os.path.join(checkpoint_dir, f)) / 1024
    print(f"  {f:<30} {size:.0f} KB")

# Create zip
shutil.make_archive("/content/neurox_checkpoints", "zip", checkpoint_dir)
zip_size = os.path.getsize("/content/neurox_checkpoints.zip") / (1024*1024)
print(f"
Checkpoints zip: {zip_size:.1f} MB")
print("
Downloading...")

# Download
files.download("/content/neurox_checkpoints.zip")
print()
print("DONE! Unzip into your trading PC:")
print("  neurox_v4/checkpoints/")


## Alternative: Save to Google Drive (if download fails)

If the direct download doesn't work (Colab can be flaky with large downloads), use Google Drive instead:


In [ ]:
#@title Alternative: Save to Google Drive
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Copy checkpoints to Drive
import shutil
dst = "/content/drive/MyDrive/neurox_v4_checkpoints"
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree("/content/Claude/NeuroX/neurox_v4/checkpoints", dst)

print(f"Checkpoints saved to Google Drive: {dst}")
print()
print("On your trading PC:")
print("  1. Open Google Drive in browser")
print("  2. Download the neurox_v4_checkpoints folder")
print("  3. Copy contents into: neurox_v4/checkpoints/")


## How to retrain (weekly)

Run this notebook once a week to keep your models fresh with latest market data:
1. Open this notebook in Colab
2. Runtime -> Run all
3. Download new checkpoints
4. Replace checkpoints/ folder on your trading PC
5. Restart your EA

The models will train on the latest 7 days of 1-minute data, 60 days of 15-minute data, and 2 years of hourly data from Yahoo Finance, ensuring they stay adapted to current market conditions.
